# 📥 Download Fyers 5-Minute OHLCV Data

Downloads historical 5-minute candle data from Fyers API and saves to **Parquet + CSV**.

**Fyers API limits:**
- Intraday (1m–30m): max **100 days per request**
- This notebook loops in 90-day chunks to stay under the limit
- Historical data available from ~2018 onwards for NSE equities

**Output files:**
- `data/{SYMBOL}_{INTERVAL}_full.parquet` — fast loading, preserves dtypes
- `data/{SYMBOL}_{INTERVAL}_full.csv` — human-readable, importable anywhere

In [ ]:
# ── Config ────────────────────────────────────────────────────────────────────
SYMBOL     = "NSE:RELIANCE-EQ"   # Fyers format: NSE:SYMBOL-EQ, NSE:NIFTY50-INDEX
INTERVAL   = "5m"                # 1m | 5m | 15m | 30m | 1h | 1D
YEARS      = 5                   # how many years back to fetch
OUTPUT_DIR = "data"              # folder to save files

# Optional: override token here (or let it load from .env)
FYERS_CLIENT_ID    = ""   # leave blank to use .env
FYERS_ACCESS_TOKEN = ""   # leave blank to use .env

In [ ]:
import sys, os
from pathlib import Path
import pandas as pd
from datetime import datetime, timezone, timedelta
import time

ROOT = Path(".").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

Path(OUTPUT_DIR).mkdir(exist_ok=True)
print("Project root:", ROOT)

In [ ]:
# ── Load credentials ──────────────────────────────────────────────────────────
from backend.config import settings

client_id    = FYERS_CLIENT_ID    or settings.FYERS_CLIENT_ID
access_token = FYERS_ACCESS_TOKEN or settings.FYERS_ACCESS_TOKEN

if not client_id or not access_token:
    raise ValueError(
        "FYERS_CLIENT_ID and FYERS_ACCESS_TOKEN must be set in .env\n"
        "Get a fresh token via: curl http://localhost:8100/fyers/login-url"
    )

print(f"Client ID   : {client_id}")
print(f"Token prefix: {access_token[:20]}...")

In [ ]:
# ── Check auth before downloading ────────────────────────────────────────────
from backend.brokers.fyers.source import FyersSource

src = FyersSource(client_id=client_id, access_token=access_token)
ok, msg = src.check_auth()

if not ok:
    raise PermissionError(
        f"Fyers auth failed: {msg}\n"
        f"Refresh token: POST http://localhost:8100/fyers/generate-token"
    )
print(f"✅ Fyers authenticated: {msg}")

In [ ]:
# ── Download in chunks ────────────────────────────────────────────────────────
# Fyers intraday limit: 100 days per request — we use 90-day chunks with 1s delay

CHUNK_DAYS  = 90
DELAY_SEC   = 1.0   # polite rate limiting between API calls

end_dt   = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
start_dt = end_dt - timedelta(days=365 * YEARS)

print(f"Fetching {SYMBOL} {INTERVAL} from {start_dt.date()} to {end_dt.date()}")
print(f"Chunk size: {CHUNK_DAYS} days  |  Estimated chunks: {int((end_dt-start_dt).days/CHUNK_DAYS)+1}")
print()

all_dfs   = []
errors    = []
current   = start_dt
chunk_num = 0

while current < end_dt:
    chunk_end = min(current + timedelta(days=CHUNK_DAYS), end_dt)
    chunk_num += 1

    try:
        df = src.fetch_ohlc(
            symbol=SYMBOL,
            from_dt=current,
            to_dt=chunk_end,
            interval=INTERVAL,
        )
        if not df.empty:
            all_dfs.append(df)
            print(f"  Chunk {chunk_num:3d}  {current.date()} → {chunk_end.date()}  rows={len(df):,}")
        else:
            print(f"  Chunk {chunk_num:3d}  {current.date()} → {chunk_end.date()}  (empty — weekend/holiday?)")

    except Exception as e:
        msg = str(e)
        print(f"  Chunk {chunk_num:3d}  {current.date()} → {chunk_end.date()}  ⚠ ERROR: {msg[:80]}")
        errors.append({"from": str(current.date()), "to": str(chunk_end.date()), "error": msg})

    current = chunk_end
    time.sleep(DELAY_SEC)

print(f"\nDone. Chunks fetched: {len(all_dfs)}  |  Errors: {len(errors)}")

In [ ]:
# ── Combine and deduplicate ───────────────────────────────────────────────────
if not all_dfs:
    raise RuntimeError("No data fetched — check symbol format, token, and date range")

full_df = pd.concat(all_dfs)
full_df = full_df[~full_df.index.duplicated(keep="first")].sort_index()

# Convert UTC index to IST for display
full_df_ist = full_df.copy()
full_df_ist.index = full_df_ist.index.tz_convert("Asia/Kolkata")

print(f"Total rows    : {len(full_df):,}")
print(f"Date range    : {full_df_ist.index[0]}  →  {full_df_ist.index[-1]}")
print(f"Columns       : {list(full_df.columns)}")
print(f"Memory usage  : {full_df.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
full_df_ist.head(10)

In [ ]:
# ── Quick stats ───────────────────────────────────────────────────────────────
print("=== OHLCV Stats ===")
print(full_df.describe().round(2))
print()

# Check for gaps (missing candles)
import numpy as np
INTERVAL_MINS = {"1m":1,"5m":5,"15m":15,"30m":30,"1h":60,"1D":1440}
mins = INTERVAL_MINS.get(INTERVAL, 5)
expected_delta = pd.Timedelta(minutes=mins)
diffs = full_df_ist.index.to_series().diff().dropna()

# Only check diffs during market hours (9:15–15:30 IST, Mon–Fri)
big_gaps = diffs[diffs > expected_delta * 3]
print(f"Candles with gap > 3x interval: {len(big_gaps)}")
if len(big_gaps) > 0 and len(big_gaps) <= 20:
    print(big_gaps.head(20))

In [ ]:
# ── Save to parquet + CSV ─────────────────────────────────────────────────────
safe_symbol = SYMBOL.replace(":", "_").replace("-", "_")
base_name   = f"{safe_symbol}_{INTERVAL}_full"

parquet_path = Path(OUTPUT_DIR) / f"{base_name}.parquet"
csv_path     = Path(OUTPUT_DIR) / f"{base_name}.csv"

full_df.to_parquet(parquet_path)
full_df_ist.to_csv(csv_path)   # save IST timestamps in CSV for readability

print(f"✅ Parquet saved : {parquet_path}  ({parquet_path.stat().st_size/1024:.0f} KB)")
print(f"✅ CSV saved     : {csv_path}  ({csv_path.stat().st_size/1024:.0f} KB)")

if errors:
    import json
    err_path = Path(OUTPUT_DIR) / f"{base_name}_errors.json"
    err_path.write_text(json.dumps(errors, indent=2))
    print(f"⚠ Error log     : {err_path}  ({len(errors)} failed chunks)")

In [ ]:
# ── Plot sample (last 3 trading days) ────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

sample = full_df_ist.tail(3 * 75)   # ~3 days × 75 candles/day (5m)

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 7), sharex=True,
                                gridspec_kw={"height_ratios": [3, 1]},
                                facecolor="#0d1117")
for ax in (ax1, ax2):
    ax.set_facecolor("#161b22")
    ax.tick_params(colors="#8b949e", labelsize=8)
    for s in ax.spines.values(): s.set_edgecolor("#30363d")

# Candlestick
for i, (ts, row) in enumerate(sample.iterrows()):
    o, h, l, c = row["open"], row["high"], row["low"], row["close"]
    col = "#3fb950" if c >= o else "#f85149"
    ax1.plot([i, i], [l, h], color=col, linewidth=0.8)
    ax1.add_patch(mpatches.Rectangle((i - 0.35, min(o, c)), 0.7, abs(c - o), color=col))

ax1.set_title(f"{SYMBOL}  {INTERVAL}  — Last 3 days", color="#c9d1d9", fontsize=12)
ax1.set_ylabel("Price", color="#8b949e")

# Volume
colors = ["#3fb950" if r["close"] >= r["open"] else "#f85149" for _, r in sample.iterrows()]
ax2.bar(range(len(sample)), sample["volume"].values, color=colors, alpha=0.7, width=0.8)
ax2.set_ylabel("Volume", color="#8b949e")

# X-axis labels — show date every ~75 candles
tick_positions = list(range(0, len(sample), 75))
tick_labels    = [str(sample.index[i].date()) for i in tick_positions]
ax2.set_xticks(tick_positions)
ax2.set_xticklabels(tick_labels, color="#8b949e")

plt.tight_layout()
plt.savefig(Path(OUTPUT_DIR) / f"{base_name}_sample.png",
            dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()
print("Chart saved.")

In [ ]:
# ── How to reload the data ────────────────────────────────────────────────────
print("To reload the data later:")
print()
print("  import pandas as pd")
print(f"  df = pd.read_parquet('{parquet_path}')  # UTC index")
print(f"  df = pd.read_csv('{csv_path}', index_col=0, parse_dates=True)  # IST index")
print()
print("To use with kronos_train.ipynb:")
print(f"  raw_df = pd.read_parquet('{parquet_path}')")
print(f"  # Set SYMBOL='{SYMBOL}', INTERVAL='{INTERVAL}' in kronos_train.ipynb")
print(f"  # Skip the fetch_all() cell and replace with: raw_df = pd.read_parquet(cache_path)")

In [ ]:
# ── Download multiple symbols in one run ─────────────────────────────────────
# Uncomment and run this cell to batch-download several symbols at once

# SYMBOLS_TO_DOWNLOAD = [
#     "NSE:RELIANCE-EQ",
#     "NSE:TCS-EQ",
#     "NSE:HDFCBANK-EQ",
#     "NSE:ICICIBANK-EQ",
#     "NSE:SBIN-EQ",
#     "NSE:NIFTY50-INDEX",
#     "NSE:NIFTYBANK-INDEX",
# ]
# 
# for sym in SYMBOLS_TO_DOWNLOAD:
#     print(f"\n=== {sym} ===")
#     safe = sym.replace(":", "_").replace("-", "_")
#     out  = Path(OUTPUT_DIR) / f"{safe}_{INTERVAL}_full.parquet"
#     if out.exists():
#         print(f"  Already exists ({out}) — skipping")
#         continue
#     chunk_dfs = []
#     cur = start_dt
#     while cur < end_dt:
#         ce = min(cur + timedelta(days=CHUNK_DAYS), end_dt)
#         try:
#             df = src.fetch_ohlc(symbol=sym, from_dt=cur, to_dt=ce, interval=INTERVAL)
#             if not df.empty: chunk_dfs.append(df)
#             print(f"  {cur.date()} → {ce.date()}  rows={len(df)}")
#         except Exception as e:
#             print(f"  {cur.date()} ⚠ {e}")
#         cur = ce
#         time.sleep(DELAY_SEC)
#     if chunk_dfs:
#         merged = pd.concat(chunk_dfs)
#         merged = merged[~merged.index.duplicated(keep="first")].sort_index()
#         merged.to_parquet(out)
#         print(f"  Saved {len(merged):,} rows → {out}")
print("Batch download cell ready — uncomment to use.")